## **LoRA Fine-Tuning**

This notebook fine-tunes a small open LLM using pure LoRA (no QLoRA / no 4-bit quantization) so you can see the standard LoRA workflow clearly.

**What you'll see:**
1. GPU check (T4 = 16 GB VRAM)
2. Load a base model in FP16
3. Inspect how the *base* model answers prompts (before fine-tuning)
4. Prepare a small instruction dataset
5. Attach LoRA adapters and train
6. Inspect how the *fine-tuned* model answers the same prompts (after)
7. Save adapters

> **Base model:** `TinyLlama/TinyLlama-1.1B-Chat-v1.0` — ~1.1B params, ~2.2 GB in FP16.



## **1. Environment Setup**

In [1]:
# install the required libraries
!pip install -q \
  "transformers==5.0.0" \
  "peft==0.18.1" \
  "accelerate==1.13.0" \
  "datasets==4.8.4" \
  "trl==1.1.0" \
  "sentencepiece==0.2.1" \
  "protobuf==5.29.6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.9 MB/s eta 0:00:00


In [2]:
import sys
import torch # pre-installed in colab env

print("Python     :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)

# check GPU availability
print("CUDA avail :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name   :", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory : {props.total_memory / 1024**3:.2f} GB")

# Expected on Colab free tier: Tesla T4, ~14.5 GB

Python     : 3.12.13
PyTorch      : 2.11.0+cu128
CUDA avail : True
GPU name   : Tesla T4
GPU memory : 14.56 GB


## **2. Load the base model in FP16**

We load the model in half precision (`torch.float16`). On T4 this is the sweet spot — faster than FP32

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID) # load tokenize tool of the determined model
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# after above code lines, it pads 'PAD' tokens to make all sequences in a batch the same length
# seq1: [1, 2, 3, PAD]
# seq2: [4, 5, PAD, PAD]

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto", # add to GPU if it's available
)



model.config.use_cache = False          # required for gradient checkpointing later
model.config.pretraining_tp = 1   #=> only use a single GPU, split weights across 'n' gpu

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params/1e6:.1f} M")
print(f"Memory footprint: {model.get_memory_footprint()/1024**3:.2f} GB") #=> How many memory usage to store all params

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Total parameters: 1100.0 M
Memory footprint: 2.05 GB


## **3. Define a prompt format and test the BASE model**

We use TinyLlama's chat template. Task = text-to-SQL. Given a table schema and a question, produce a SQL query.


In [4]:
def build_prompt(schema: str, question: str) -> str:
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    # System prompt guilds invariant and fixed on any user's question
    user = f"Schema:\n{schema}\n\nQuestion: {question}"
    # schema = context, question = question
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]

    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # tokenize = False => display 'text' instead of 'token IDs'

In [5]:
# testing build_prompt function
test_prompt_1 = build_prompt(
    schema="CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
    question="List the names of employees in the Engineering department earning more than 100000."
)

test_prompt_1

'<|system|>\nYou are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.</s>\n<|user|>\nSchema:\nCREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);\n\nQuestion: List the names of employees in the Engineering department earning more than 100000.</s>\n<|assistant|>\n'

In [6]:
print(test_prompt_1)

<|system|>
You are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.</s>
<|user|>
Schema:
CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);

Question: List the names of employees in the Engineering department earning more than 100000.</s>
<|assistant|>



In [7]:
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 120) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False, # Greedy choosing, the answers will be deterministic , better in code
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    # generate output

    # Slice only newly generated tokens
    input_length = inputs["input_ids"].shape[1] # take input size
    new_tokens = output_ids[0][input_length:] # split, only take the last which is 'generate' part, output_ids include [prompt] + [generated]

    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip() # return 'texts', 'True' which means only taking the well-known tokens not special tokens like EOF


In [8]:
output = generate(test_prompt_1)
print(output)

To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE department = 'Engineering'
AND salary > 100000;
```

This query will return a list of names of employees in the Engineering department who have a salary greater than 100,000.


In [9]:
# Three probe prompts we will reuse AFTER fine-tuning for direct comparison.
PROBES = [
    {
        "schema":  "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
        "question":"List the names of employees in the Engineering department earning more than 100000.",
    },
    {
        "schema":  "CREATE TABLE orders (order_id INT, customer_id INT, amount FLOAT, order_date DATE);",
        "question":"What is the total order amount per customer in 2024?",
    },
    {
        "schema":  "CREATE TABLE movies (title TEXT, year INT, rating FLOAT, genre TEXT);",
        "question":"Show the top 5 highest rated horror movies released after 2015.",
    },
]

print("="*70)
print("BASE MODEL (before fine-tuning)")
print("="*70)
base_outputs = []
for i, p in enumerate(PROBES, 1):
    prompt = build_prompt(p["schema"], p["question"])
    ans = generate(prompt)
    base_outputs.append(ans)
    print(f"\n--- Probe {i} ---")
    print("Q:", p["question"])
    print("A:", ans)


BASE MODEL (before fine-tuning)

--- Probe 1 ---
Q: List the names of employees in the Engineering department earning more than 100000.
A: To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE department = 'Engineering'
AND salary > 100000;
```

This query will return a list of names of employees in the Engineering department who have a salary greater than 100,000.

--- Probe 2 ---
Q: What is the total order amount per customer in 2024?
A: To answer the question, you need to use the `SUM` function to calculate the total order amount per customer in 2024. Here's the SQL query:

```sql
SELECT SUM(amount) AS total_order_amount_per_customer
FROM orders
WHERE year(order_date) = 2024
GROUP BY customer_id;
```

This query will return a single row with the total order amount per customer in 2024.

--- Probe 3 ---
Q: Show the top 5 highest rated horror movies released after 2015.
A: To show the top 5 highest rated horror movies released after 2015, 

Expect the base model to *talk about* the query, add commentary, re-explain the schema, or produce malformed SQL. That is the "before" state.

## **4. Load and prepare the dataset**

We use `b-mc2/sql-create-context` — a compact text-to-SQL dataset with `(question, context, answer)` triples. We'll take a small slice so training finishes in a few minutes on T4.

In [10]:
from datasets import load_dataset

raw = load_dataset("b-mc2/sql-create-context", split="train") # ID dataset
print("Full dataset size:", len(raw))
print("Example row     :", raw[0])

# Keep it small for a fast, visible demo on T4
raw = raw.shuffle(seed=42).select(range(10000)) # Choose only 3000 samples
split = raw.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"] # train/test
print("Train:", len(train_ds), " Eval:", len(eval_ds))

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Full dataset size: 78577
Example row     : {'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}
Train: 9500  Eval: 500


In [11]:
train_ds[0]

{'answer': 'SELECT SUM(lane) FROM table_name_58 WHERE nationality = "iceland"',
 'question': 'How many lanes have a Nationality of iceland?',
 'context': 'CREATE TABLE table_name_58 (lane INTEGER, nationality VARCHAR)'}

In [12]:
train_ds[:3]

{'answer': ['SELECT SUM(lane) FROM table_name_58 WHERE nationality = "iceland"',
  'SELECT record FROM table_name_24 WHERE game = 24',
  'SELECT title FROM table_name_50 WHERE production_code = "ad1b04"'],
 'question': ['How many lanes have a Nationality of iceland?',
  'What is Record, when Game is "24"?',
  'What is the title of the episode with a production code of ad1b04?'],
 'context': ['CREATE TABLE table_name_58 (lane INTEGER, nationality VARCHAR)',
  'CREATE TABLE table_name_24 (record VARCHAR, game VARCHAR)',
  'CREATE TABLE table_name_50 (title VARCHAR, production_code VARCHAR)']}

In [13]:
print("Schema:", train_ds[0]["context"])
print("="*70)
print("Question:", train_ds[0]["question"])
print("="*70)
print("Answer:", train_ds[0]["answer"])
print("="*70)

Schema: CREATE TABLE table_name_58 (lane INTEGER, nationality VARCHAR)
Question: How many lanes have a Nationality of iceland?
Answer: SELECT SUM(lane) FROM table_name_58 WHERE nationality = "iceland"


In [14]:
def format_example(row):
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    user      = f"Schema:\n{row['context']}\n\nQuestion: {row['question']}"
    assistant = row["answer"]
    messages = [
        {"role": "system",    "content": system},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": assistant},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False) # apply template here
    return {"text": text}

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds  = eval_ds.map (format_example, remove_columns=eval_ds.column_names)

print("\n--- Formatted training example ---\n")
print(train_ds[0]["text"][:800])# understand the dataset
train_ds[0].keys()

Map:   0%|          | 0/9500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]


--- Formatted training example ---

<|system|>
You are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.</s>
<|user|>
Schema:
CREATE TABLE table_name_58 (lane INTEGER, nationality VARCHAR)

Question: How many lanes have a Nationality of iceland?</s>
<|assistant|>
SELECT SUM(lane) FROM table_name_58 WHERE nationality = "iceland"</s>



dict_keys(['text'])

In [15]:
print(train_ds[0])

{'text': '<|system|>\nYou are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.</s>\n<|user|>\nSchema:\nCREATE TABLE table_name_58 (lane INTEGER, nationality VARCHAR)\n\nQuestion: How many lanes have a Nationality of iceland?</s>\n<|assistant|>\nSELECT SUM(lane) FROM table_name_58 WHERE nationality = "iceland"</s>\n'}


## **5. Attach LoRA Adapters**

LoRA inserts low-rank trainable matrices into specific linear layers (attention projections) while freezing the base weights. For Llama-family models the common target modules are `q_proj`, `k_proj`, `v_proj`, `o_proj` (and optionally the MLP projections).

In [25]:
from peft import LoraConfig, get_peft_model

# Enable gradient checkpointing to save VRAM during backward pass
model.gradient_checkpointing_enable() # NO NEED TO STORE ALL ACTIVATIONS, only store a necessary subset
model.enable_input_require_grads()     # needed because base params are frozen

lora_config = LoraConfig(
    r=32,                                # rank
    lora_alpha=64,                       # scaling factor (alpha / r = 2.0)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# W_adapted = W + (B @ A)  # LoRA update
# b_adapted = b + Δb       # Bias update => Bias = None
# set Causal_LM to make 'masking for decoder' for case such as GPT, BERT, Llama


model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# You should see something like: trainable: ~4M / all: ~1.1B  (< 0.5%)

trainable params: 9,011,200 || all params: 1,109,059,584 || trainable%: 0.8125


## **6. Train**

We use `SFTTrainer` from TRL.

Settings chosen for T4 (16 GB):
- `per_device_train_batch_size=2`, `gradient_accumulation_steps=8` → effective batch 16
- `max_seq_length=512`
- `fp16=True` (T4 supports FP16, not BF16)
- 1 epoch over 3000 examples ≈ ~5–8 minutes


In [17]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./tinyllama-sql-lora"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2, # I change here
    per_device_train_batch_size=2, # device * per_device_train_batch_size * gradient_accumulation_steps = 2 * 1 * 8  = 'effective batch size'
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8, #
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    report_to="none",
)

# Pre-tokenize so we don't depend on SFTTrainer's text-field handling => so we don't need to decode raw texts to token IDs
def tokenize(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )
    return out

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_tok  = eval_ds.map (tokenize, batched=True, remove_columns=["text"])

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_tok,
    eval_dataset=eval_tok, #
    processing_class=tokenizer,
)

trainer.train()

Map:   0%|          | 0/9500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
50,0.642310,0.633758
100,0.608504,0.579649
150,0.561947,0.558169
200,0.567643,0.543965
250,0.551448,0.532194
300,0.533232,0.524319
350,0.514003,0.518521
400,0.519999,0.512916
450,0.528556,0.508263
500,0.531233,0.504535


TrainOutput(global_step=1188, training_loss=0.5394237573299344, metrics={'train_runtime': 3192.0064, 'train_samples_per_second': 5.952, 'train_steps_per_second': 0.372, 'total_flos': 1.6339228535046144e+16, 'train_loss': 0.5394237573299344})

In [19]:
# Peak VRAM used during training — should stay under T4's 15.8 GB
import torch
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

Peak GPU memory allocated: 2.37 GB


### **SFTConfig Parameters Explained**

- `output_dir=OUTPUT_DIR`  
  Folder where the fine-tuned model, checkpoints, and logs will be saved.

- `num_train_epochs=1`  
  Number of times the model sees the full training dataset.

- `per_device_train_batch_size=2`  
  Number of training samples processed at once on each device.

- `per_device_eval_batch_size=2`  
  Number of evaluation samples processed at once on each device.

- `gradient_accumulation_steps=8`  
  Accumulates gradients for 8 steps before updating weights, which gives an effective larger batch size.

- `gradient_checkpointing=True`  
  Saves GPU memory by recomputing some activations during backpropagation.

- `learning_rate=2e-4`  
  Controls how big each weight update is during training.

- `lr_scheduler_type="cosine"`  
  Gradually changes the learning rate using a cosine decay schedule.

- `warmup_steps=10`  
  Slowly increases the learning rate for the first 10 steps to make training stable.

- `optim="adamw_torch"`  
  Optimizer used to update model weights, here it is AdamW from PyTorch.

- `fp16=True`  
  Uses 16-bit floating point precision to reduce memory usage and speed up training.

- `bf16=False`  
  Disables bfloat16 precision.

- `logging_steps=10`  
  Logs training details every 10 steps.

- `eval_strategy="steps"`  
  Runs evaluation during training based on step intervals.

- `eval_steps=50`  
  Evaluates the model every 50 steps.

- `save_strategy="epoch"`  
  Saves a checkpoint after each epoch.

- `report_to="none"`  
  Disables reporting to tools like WandB or TensorBoard.

## **7. Compare: AFTER Fine-Tuning**

In [20]:
# Re-enable cache for fast inference
model.config.use_cache = True
model.eval()

print("="*70)
print("FINE-TUNED MODEL (after LoRA)")
print("="*70)
for i, p in enumerate(PROBES, 1):
    prompt = build_prompt(p["schema"], p["question"])
    ans = generate(prompt)
    print(f"\n--- Probe {i} ---")
    print("Q:     ", p["question"])
    print("BEFORE:", base_outputs[i-1])
    print("="*70)
    print("AFTER :", ans)
    print("="*70)


FINE-TUNED MODEL (after LoRA)

--- Probe 1 ---
Q:      List the names of employees in the Engineering department earning more than 100000.
BEFORE: To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE department = 'Engineering'
AND salary > 100000;
```

This query will return a list of names of employees in the Engineering department who have a salary greater than 100,000.
AFTER : SELECT T1.name FROM employees AS T1 JOIN departments AS T2 ON T1.department = T2.id WHERE T2.name = "Engineering" AND T1.salary > 100000

--- Probe 2 ---
Q:      What is the total order amount per customer in 2024?
BEFORE: To answer the question, you need to use the `SUM` function to calculate the total order amount per customer in 2024. Here's the SQL query:

```sql
SELECT SUM(amount) AS total_order_amount_per_customer
FROM orders
WHERE year(order_date) = 2024
GROUP BY customer_id;
```

This query will return a single row with the total order amount per customer 

You should now see:
- **Before:** verbose, chatty, often wrong syntax, re-explains the schema.
- **After:** terse, well-formed SQL — the model learned the response *style* of the dataset.

That style shift is exactly what LoRA buys you for a few minutes of T4 compute.

## **8. Save the Adapters**

LoRA adapters are tiny (a few MB). You save *only* the adapter, not the whole base model.

In [21]:
ADAPTER_DIR = "./tinyllama-sql-lora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import os
total = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f"Adapter size on disk: {total/1024**2:.2f} MB")

Adapter size on disk: 12.07 MB


In [22]:
import shutil

zip_file_name = f'{ADAPTER_DIR}.zip'
shutil.make_archive(base_name=ADAPTER_DIR, format='zip', root_dir=ADAPTER_DIR)
print(f"Created {zip_file_name}")

Created ./tinyllama-sql-lora-adapter.zip


In [23]:
from google.colab import files

files.download(zip_file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **Loading the adapter later (for reference)**

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base, "./tinyllama-sql-lora-adapter")
tokenizer = AutoTokenizer.from_pretrained("./tinyllama-sql-lora-adapter")
```


## Recap

| Step | What happened |
|---|---|
| Base model loaded in FP16 | ~2.2 GB VRAM |
| LoRA adapters attached (r=16, target = attention projections) | ~4M trainable params |
| Trained 1 epoch on 3000 SQL examples | ~5–8 min on T4, peak VRAM well under 16 GB |
| Adapter saved | A few MB on disk |
| Visible outcome | Chatty base → clean SQL after fine-tuning |

**To extend this:** swap the dataset (e.g. `databricks/databricks-dolly-15k`, a style-transfer set, or your own JSONL), keep the same LoRA config, and you've got the same workflow for any instruction-following task that fits in T4.


---

**Note on accuracy**

This notebook is a demonstration of the LoRA workflow, not a production SQL model. With a small 1.1B base, 3,000 examples, and 1 epoch, the fine-tuned model reliably learns the response style (clean SQL instead of chatty explanations) but is not always semantically correct. Style learning ≠ task mastery.
To improve accuracy, the same workflow scales up: use a larger base model (e.g. Qwen2.5-3B, or a 7B with QLoRA), more training data (the full dataset has ~78k examples), and 2–3 epochs instead of 1. The code stays the same; only the scale changes.

---